In [ ]:
import os

### Write ```Dockerfile```

In [ ]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

### Write ```requirements.txt``` to local drive

In [ ]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
catboost==1.0.4
scikit_learn==0.24.1

### Write ```script.py``` to local drive

In [ ]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import catboost as cb
import sklearn.metrics as skm

# get metric
def get_metric_by_year_month(target, y_hat, str_eval_metric):
    if str_eval_metric == 'AUC':
        return skm.roc_auc_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'PRAUC':
        return skm.average_precision_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'Logloss':
        return skm.log_loss(y_true=target, y_pred=y_hat)
    elif str_eval_metric == 'F1':
        return skm.f1_score(y_true=target, y_pred=y_hat)
    else:
        pass

# get index of array
int_idx_array = int(os.environ['AWS_BATCH_JOB_ARRAY_INDEX'])
print(f'Array index: {int_idx_array}')

# constants
str_project = '20240509-christian-internship'
str_target = 'TARGET'

###############################################################################
# HYPERPARAMETERS
###############################################################################

# get number of iterations
int_n_iterations = 100
print(f'Iterations: {int_n_iterations}')

# get filename for training
str_filename_train = 'df_train.gzip'
print(f'Training filename: {str_filename_train}')

# get filename for valid
str_filename_valid = 'df_valid.gzip'
print(f'Valid filename: {str_filename_valid}')

# number of tuning jobs
int_n_tuning_jobs = 10
print(f'Total tuning jobs: {int_n_tuning_jobs}')

# get proportion of iterations to use as early stopping rounds
flt_prop_early_stopping = 0.05
print(f'Proportion early stopping: {flt_prop_early_stopping}')

# get eval metric
str_eval_metric = 'AUC'
print(f'Eval metric: {str_eval_metric}')

##################################################################################

# get lr
print('Getting learning rate...')
flt_learning_rate = list(np.around(np.linspace(0.001, 0.999, int_n_tuning_jobs), 4))[int_idx_array] # round each value to 4 decimal places

# read training data
print('Reading training data...')
str_uri = f's3://{str_project}/09_aws_batch/01_create_image/{str_filename_train}'
df = pd.read_parquet(str_uri)

# make sure all cols in list_cols_model are in df
list_cols_model = [col for col in df.columns if col != str_target]

# get the non numeric feats
print('Getting list of non-numeric columns...')
list_cols_non_numeric = []
for col in list_cols_model:
    if df[col].dtype not in ['float64','int64']:
        list_cols_non_numeric.append(col)

# build model
print('Building model...')
# pool data
pool_train = cb.Pool(
    df[list_cols_model], 
    df[str_target], 
    cat_features=list_cols_non_numeric,
)
del df

# read validation data
str_uri = f's3://{str_project}/09_aws_batch/01_create_image/{str_filename_valid}'
df = pd.read_parquet(str_uri)

# pool data
pool_valid = cb.Pool(
    df[list_cols_model], 
    df[str_target], 
    cat_features=list_cols_non_numeric,
)
del df

# init class
cls_model_inference = cb.CatBoostClassifier(
    task_type='CPU',
    nan_mode='Min',
    random_state=42,
    eval_metric=str_eval_metric,
    iterations=int_n_iterations,
    learning_rate=flt_learning_rate,
    #class_weights=list_class_weights,
    #monotone_constraints=dict_monotone_constraints,
)

# fit
cls_model_inference.fit(
    pool_train,
    eval_set=[pool_valid],
    verbose=10,
    use_best_model=True,
    early_stopping_rounds=int(round(int_n_iterations*flt_prop_early_stopping)), 
)
del pool_train
del pool_valid

################################################################################################
# GET TRAINING EVAL METRIC
################################################################################################
print('Getting training eval metric...')

# import data
str_uri = f's3://{str_project}/09_aws_batch/01_create_image/{str_filename_train}'
df = pd.read_parquet(str_uri)

# get predictions
if str_eval_metric in ['AUC','PRAUC','Logloss']:
    # probabilities
    df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
elif str_eval_metric in ['F1']:
    # class
    df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
else:
    pass

# get eval metric - train
if str_eval_metric == 'AUC':
    flt_eval_metric_train = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'PRAUC':
    flt_eval_metric_train = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'Logloss':
    flt_eval_metric_train = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
elif str_eval_metric == 'F1':
    flt_eval_metric_train = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

# save memory
del df

################################################################################################
# GET VALIDATION EVAL METRIC (WEIGHTED)
################################################################################################
print('Getting validation eval metric...')

# import data
str_uri = f's3://{str_project}/09_aws_batch/01_create_image/{str_filename_valid}'
df = pd.read_parquet(str_uri)

# get predictions
if str_eval_metric in ['AUC','PRAUC','Logloss']:
    # probabilities
    df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
elif str_eval_metric in ['F1']:
    # class
    df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
else:
    pass

# get eval metric - valid
if str_eval_metric == 'AUC':
    flt_eval_metric_valid = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'PRAUC':
    flt_eval_metric_valid = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'Logloss':
    flt_eval_metric_valid = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
elif str_eval_metric == 'F1':
    flt_eval_metric_valid = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

# save memory
del df

################################################################################################
# CREATE OUTPUT DATA FRAME
################################################################################################
print('Creating output data frame...')

flt_diff = abs(flt_eval_metric_train - flt_eval_metric_valid)
dict_row = {
    'iteration': int_idx_array,
    'learning_rate': flt_learning_rate,
    'flt_eval_metric_train': flt_eval_metric_train,
    'flt_eval_metric_valid': flt_eval_metric_valid,
    'diff': flt_diff,
    'best_iteration': cls_model_inference.get_best_iteration(),
}
df = pd.DataFrame(dict_row, index=[0])

# save
str_filename = f'df_output_{int_idx_array}.csv'
str_uri = f's3://{str_project}/09_aws_batch/output/{str_filename}'
df.to_csv(str_uri, index=False)

### Build and push to ECR

In [ ]:
%%sh

# name the image
image=christian-tuning

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

### Clean-up

In [ ]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass